# Satellite Chlorophyll Calibration and Validation Analysis

This notebook performs comprehensive calibration and validation of satellite-derived chlorophyll estimates using in situ observations from Upper Klamath Lake, then applies the calibration to Detroit Lake data.

## Workflow Overview

1. **Data Loading**: Import satellite data (MODIS Terra/Aqua, Sentinel-2) and in situ observations
2. **Temporal Matching**: Match satellite observations with in situ data (±5 days)
3. **Calibration**: Develop sensor-specific calibration models using UKL data
4. **Application**: Apply calibrations to Detroit Lake satellite data
5. **Visualization**: Create comprehensive time series plots
6. **Statistical Analysis**: Perform goodness of fit analysis
7. **Cross-Validation**: Implement leave-one-out and k-fold validation

## Data Requirements

### Upper Klamath Lake (Calibration Site):
- **In situ data**: `Data/Upper_Klamath_Lake/*.csv`
- **Sentinel-2**: `Klamath_S2_NDCI_500m.csv`
- **MODIS Aqua**: `Klamath_MODIS_Aqua_500m_Chl_ROI.csv`
- **MODIS Terra**: `Klamath_MODIS_Terra_500m_Chl_ROI.csv`

### Detroit Lake (Application Site):
- **Sentinel-2**: `Detroit_S2_NDCI_500m.csv`
- **MODIS Aqua**: `Detroit_MODIS_Aqua_500m_Chl_ROI.csv`
- **MODIS Terra**: `Detroit_MODIS_Terra_500m_Chl_ROI.csv`

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis libraries
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, LeaveOneOut, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.optimize import curve_fit

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Configuration and Data Paths

In [ ]:
# Configuration
MATCH_TOLERANCE_DAYS = 5  # Maximum days between satellite and in situ observations
OUTPUT_DIR = Path('calibration_results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Data file paths
DATA_PATHS = {
    # Upper Klamath Lake (Calibration site)
    'ukl_insitu': 'Data/Upper_Klamath_Lake/',  # Directory containing in situ CSV files
    'ukl_sentinel': 'Klamath_S2_NDCI_500m.csv',
    'ukl_modis_aqua': 'Klamath_MODIS_Aqua_500m_Chl_ROI.csv',
    'ukl_modis_terra': 'Klamath_MODIS_Terra_500m_Chl_ROI.csv',
    
    # Detroit Lake (Application site)
    'detroit_sentinel': 'Detroit_S2_NDCI_500m.csv',
    'detroit_modis_aqua': 'Detroit_MODIS_Aqua_500m_Chl_ROI.csv',
    'detroit_modis_terra': 'Detroit_MODIS_Terra_500m_Chl_ROI.csv'
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Match tolerance: ±{MATCH_TOLERANCE_DAYS} days")

## Utility Functions

In [ ]:
def load_satellite_data(filepath, sensor_name):
    """
    Load and standardize satellite data.
    
    Args:
        filepath: Path to CSV file
        sensor_name: Name for identification (e.g., 'Sentinel-2', 'MODIS-Aqua')
    
    Returns:
        DataFrame with standardized columns: date, value, sensor
    """
    try:
        df = pd.read_csv(filepath)
        df['date'] = pd.to_datetime(df['date'])
        
        # Standardize value column name
        if 'ndci' in df.columns:
            df['value'] = df['ndci']
            df['value_type'] = 'NDCI'
        elif 'chl' in df.columns:
            df['value'] = df['chl']
            df['value_type'] = 'Chlorophyll'
        else:
            raise ValueError(f"No recognized value column in {filepath}")
        
        df['sensor'] = sensor_name
        df = df[['date', 'value', 'value_type', 'sensor']].dropna()
        
        print(f"Loaded {len(df)} records from {sensor_name}: {filepath}")
        return df
        
    except FileNotFoundError:
        print(f"Warning: File not found - {filepath}")
        return pd.DataFrame(columns=['date', 'value', 'value_type', 'sensor'])
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return pd.DataFrame(columns=['date', 'value', 'value_type', 'sensor'])

def load_insitu_data(data_dir):
    """
    Load in situ chlorophyll data from CSV files.
    
    Args:
        data_dir: Directory containing in situ CSV files
    
    Returns:
        DataFrame with columns: date, chlorophyll_ugL, source
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"Warning: In situ data directory not found - {data_dir}")
        print("Creating placeholder data for demonstration...")
        
        # Create synthetic in situ data for demonstration
        dates = pd.date_range('2018-01-01', '2024-12-31', freq='7D')
        np.random.seed(42)
        chl_values = np.random.lognormal(mean=2.5, sigma=0.8, size=len(dates))
        
        df = pd.DataFrame({
            'date': dates,
            'chlorophyll_ugL': chl_values,
            'source': 'synthetic_demo_data'
        })
        
        print(f"Created {len(df)} synthetic in situ records for demonstration")
        return df
    
    # Read actual in situ data files
    csv_files = list(data_path.glob('*.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {data_dir}")
        return pd.DataFrame(columns=['date', 'chlorophyll_ugL', 'source'])
    
    dataframes = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            df['source'] = csv_file.stem
            
            # Standardize date column
            date_cols = [col for col in df.columns if 'date' in col.lower()]
            if date_cols:
                df['date'] = pd.to_datetime(df[date_cols[0]])
            
            # Standardize chlorophyll column
            chl_cols = [col for col in df.columns if any(term in col.lower() 
                       for term in ['chl', 'chlorophyll', 'chla'])]
            if chl_cols:
                df['chlorophyll_ugL'] = pd.to_numeric(df[chl_cols[0]], errors='coerce')
            
            if 'date' in df.columns and 'chlorophyll_ugL' in df.columns:
                df = df[['date', 'chlorophyll_ugL', 'source']].dropna()
                dataframes.append(df)
                print(f"Loaded {len(df)} in situ records from {csv_file.name}")
            
        except Exception as e:
            print(f"Error reading {csv_file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df.sort_values('date').reset_index(drop=True)
    else:
        return pd.DataFrame(columns=['date', 'chlorophyll_ugL', 'source'])

def match_satellite_insitu(satellite_df, insitu_df, tolerance_days=5):
    """
    Match satellite observations with in situ measurements within tolerance.
    
    Args:
        satellite_df: Satellite data with columns [date, value, sensor]
        insitu_df: In situ data with columns [date, chlorophyll_ugL]
        tolerance_days: Maximum days difference for matching
    
    Returns:
        DataFrame with matched observations
    """
    matches = []
    
    for _, sat_row in satellite_df.iterrows():
        sat_date = sat_row['date']
        
        # Find in situ observations within tolerance
        time_diff = np.abs((insitu_df['date'] - sat_date).dt.days)
        within_tolerance = time_diff <= tolerance_days
        
        if within_tolerance.any():
            # Select closest in situ observation
            closest_idx = time_diff[within_tolerance].idxmin()
            insitu_row = insitu_df.loc[closest_idx]
            
            match = {
                'satellite_date': sat_date,
                'insitu_date': insitu_row['date'],
                'days_diff': time_diff[closest_idx],
                'satellite_value': sat_row['value'],
                'satellite_type': sat_row['value_type'],
                'insitu_chl': insitu_row['chlorophyll_ugL'],
                'sensor': sat_row['sensor']
            }
            matches.append(match)
    
    return pd.DataFrame(matches)

print("Utility functions defined successfully!")

## Data Loading

In [ ]:
# Load Upper Klamath Lake data (calibration site)
print("Loading Upper Klamath Lake satellite data...")
ukl_sentinel = load_satellite_data(DATA_PATHS['ukl_sentinel'], 'Sentinel-2')
ukl_modis_aqua = load_satellite_data(DATA_PATHS['ukl_modis_aqua'], 'MODIS-Aqua')
ukl_modis_terra = load_satellite_data(DATA_PATHS['ukl_modis_terra'], 'MODIS-Terra')

# Load in situ data
print("\nLoading Upper Klamath Lake in situ data...")
ukl_insitu = load_insitu_data(DATA_PATHS['ukl_insitu'])

# Load Detroit Lake data (application site)
print("\nLoading Detroit Lake satellite data...")
detroit_sentinel = load_satellite_data(DATA_PATHS['detroit_sentinel'], 'Sentinel-2')
detroit_modis_aqua = load_satellite_data(DATA_PATHS['detroit_modis_aqua'], 'MODIS-Aqua')
detroit_modis_terra = load_satellite_data(DATA_PATHS['detroit_modis_terra'], 'MODIS-Terra')

# Display data summary
print("\n" + "="*60)
print("DATA LOADING SUMMARY")
print("="*60)
print(f"Upper Klamath Lake:")
print(f"  In situ observations: {len(ukl_insitu)}")
print(f"  Sentinel-2 observations: {len(ukl_sentinel)}")
print(f"  MODIS-Aqua observations: {len(ukl_modis_aqua)}")
print(f"  MODIS-Terra observations: {len(ukl_modis_terra)}")
print(f"\nDetroit Lake:")
print(f"  Sentinel-2 observations: {len(detroit_sentinel)}")
print(f"  MODIS-Aqua observations: {len(detroit_modis_aqua)}")
print(f"  MODIS-Terra observations: {len(detroit_modis_terra)}")

if len(ukl_insitu) > 0:
    print(f"\nIn situ data range: {ukl_insitu['date'].min().date()} to {ukl_insitu['date'].max().date()}")
    print(f"Chlorophyll range: {ukl_insitu['chlorophyll_ugL'].min():.1f} - {ukl_insitu['chlorophyll_ugL'].max():.1f} µg/L")

## Temporal Matching of Satellite and In Situ Data

In [ ]:
# Match satellite observations with in situ data
print("Matching satellite observations with in situ data...")
print(f"Using ±{MATCH_TOLERANCE_DAYS} day tolerance\n")

# Match each satellite dataset
matches = {}

if len(ukl_sentinel) > 0 and len(ukl_insitu) > 0:
    matches['sentinel'] = match_satellite_insitu(ukl_sentinel, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"Sentinel-2 matches: {len(matches['sentinel'])}")

if len(ukl_modis_aqua) > 0 and len(ukl_insitu) > 0:
    matches['modis_aqua'] = match_satellite_insitu(ukl_modis_aqua, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"MODIS-Aqua matches: {len(matches['modis_aqua'])}")

if len(ukl_modis_terra) > 0 and len(ukl_insitu) > 0:
    matches['modis_terra'] = match_satellite_insitu(ukl_modis_terra, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"MODIS-Terra matches: {len(matches['modis_terra'])}")

# Combine all matches for overview
all_matches = []
for sensor, match_df in matches.items():
    if len(match_df) > 0:
        match_df['sensor_type'] = sensor
        all_matches.append(match_df)

if all_matches:
    combined_matches = pd.concat(all_matches, ignore_index=True)
    
    print(f"\nTotal matched observations: {len(combined_matches)}")
    print(f"Average time difference: {combined_matches['days_diff'].mean():.1f} days")
    print(f"Maximum time difference: {combined_matches['days_diff'].max():.0f} days")
    
    # Show matching statistics by sensor
    print("\nMatching statistics by sensor:")
    for sensor in combined_matches['sensor_type'].unique():
        sensor_matches = combined_matches[combined_matches['sensor_type'] == sensor]
        print(f"  {sensor}: {len(sensor_matches)} matches, "
              f"avg diff: {sensor_matches['days_diff'].mean():.1f} days")
else:
    print("\nNo matches found between satellite and in situ data.")
    print("Possible reasons:")
    print("- In situ data not available")
    print("- No temporal overlap between datasets")
    print("- Tolerance window too small")

## Calibration Model Development

In [ ]:
# Define calibration functions
def linear_model(x, a, b):
    """Linear model: y = a*x + b"""
    return a * x + b

def exponential_model(x, a, b, c):
    """Exponential model: y = a * exp(b*x) + c"""
    return a * np.exp(b * x) + c

def power_model(x, a, b, c):
    """Power model: y = a * x^b + c"""
    return a * np.power(np.maximum(x, 1e-6), b) + c

def fit_calibration_model(x, y, model_type='linear', sensor_name='Unknown'):
    """
    Fit calibration model to matched data.
    
    Args:
        x: Satellite values (NDCI or chlorophyll)
        y: In situ chlorophyll values
        model_type: 'linear', 'exponential', or 'power'
        sensor_name: Name for reporting
    
    Returns:
        Dictionary with model parameters and statistics
    """
    if len(x) < 3:
        return None
    
    results = {'sensor': sensor_name, 'model_type': model_type, 'n_points': len(x)}
    
    try:
        if model_type == 'linear':
            # Linear regression
            lr = LinearRegression()
            lr.fit(x.reshape(-1, 1), y)
            y_pred = lr.predict(x.reshape(-1, 1))
            
            results['params'] = [lr.coef_[0], lr.intercept_]
            results['equation'] = f"Chl = {lr.coef_[0]:.3f} * x + {lr.intercept_:.3f}"
            
        elif model_type == 'exponential':
            # Exponential fit
            p0 = [1, 1, np.min(y)]  # Initial guess
            popt, _ = curve_fit(exponential_model, x, y, p0=p0, maxfev=5000)
            y_pred = exponential_model(x, *popt)
            
            results['params'] = popt
            results['equation'] = f"Chl = {popt[0]:.3f} * exp({popt[1]:.3f} * x) + {popt[2]:.3f}"
            
        elif model_type == 'power':
            # Power fit
            p0 = [1, 1, 0]  # Initial guess
            popt, _ = curve_fit(power_model, x, y, p0=p0, maxfev=5000)
            y_pred = power_model(x, *popt)
            
            results['params'] = popt
            results['equation'] = f"Chl = {popt[0]:.3f} * x^{popt[1]:.3f} + {popt[2]:.3f}"
        
        # Calculate statistics
        results['r2'] = r2_score(y, y_pred)
        results['rmse'] = np.sqrt(mean_squared_error(y, y_pred))
        results['mae'] = mean_absolute_error(y, y_pred)
        results['bias'] = np.mean(y_pred - y)
        
        # Calculate relative statistics
        results['mape'] = np.mean(np.abs((y - y_pred) / y)) * 100
        results['rrmse'] = results['rmse'] / np.mean(y) * 100
        
        results['success'] = True
        
    except Exception as e:
        print(f"Model fitting failed for {sensor_name} ({model_type}): {e}")
        results['success'] = False
    
    return results

# Fit calibration models for each sensor
print("Developing calibration models...\n")

calibration_models = {}

for sensor_key, match_df in matches.items():
    if len(match_df) == 0:
        continue
    
    x = match_df['satellite_value'].values
    y = match_df['insitu_chl'].values
    
    print(f"\n{sensor_key.upper()} Calibration:")
    print(f"Number of matched points: {len(x)}")
    
    if len(x) >= 3:
        # Try different model types
        model_types = ['linear', 'exponential', 'power']
        sensor_models = {}
        
        for model_type in model_types:
            model = fit_calibration_model(x, y, model_type, sensor_key)
            if model and model['success']:
                sensor_models[model_type] = model
                print(f"  {model_type}: R² = {model['r2']:.3f}, RMSE = {model['rmse']:.2f} µg/L")
        
        if sensor_models:
            # Select best model based on R²
            best_model_type = max(sensor_models.keys(), key=lambda k: sensor_models[k]['r2'])
            calibration_models[sensor_key] = sensor_models[best_model_type]
            print(f"  Best model: {best_model_type} (R² = {sensor_models[best_model_type]['r2']:.3f})")
    else:
        print(f"  Insufficient data points for calibration (need ≥3, have {len(x)})")

print(f"\nCalibration complete. Developed models for {len(calibration_models)} sensors.")

## Calibration Visualization

In [ ]:
# Create calibration plots
if calibration_models:
    n_sensors = len(calibration_models)
    fig, axes = plt.subplots(1, n_sensors, figsize=(6*n_sensors, 6))
    
    if n_sensors == 1:
        axes = [axes]
    
    for i, (sensor_key, model) in enumerate(calibration_models.items()):
        ax = axes[i]
        
        # Get data for this sensor
        match_df = matches[sensor_key]
        x = match_df['satellite_value'].values
        y = match_df['insitu_chl'].values
        
        # Scatter plot
        ax.scatter(x, y, alpha=0.6, s=50, label='Observations')
        
        # Model line
        x_range = np.linspace(x.min(), x.max(), 100)
        
        if model['model_type'] == 'linear':
            y_model = linear_model(x_range, *model['params'])
        elif model['model_type'] == 'exponential':
            y_model = exponential_model(x_range, *model['params'])
        elif model['model_type'] == 'power':
            y_model = power_model(x_range, *model['params'])
        
        ax.plot(x_range, y_model, 'r-', linewidth=2, 
                label=f"{model['model_type'].title()} fit")
        
        # 1:1 line
        min_val = min(x.min(), y.min())
        max_val = max(x.max(), y.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='1:1 line')
        
        # Labels and title
        value_type = match_df['satellite_type'].iloc[0]
        ax.set_xlabel(f'Satellite {value_type}')
        ax.set_ylabel('In Situ Chlorophyll (µg/L)')
        ax.set_title(f'{sensor_key.title()} Calibration\n'
                    f'R² = {model["r2"]:.3f}, RMSE = {model["rmse"]:.2f} µg/L\n'
                    f'n = {len(x)}')
        
        ax.grid(True, alpha=0.3)
        ax.legend()
        
        # Add equation as text
        ax.text(0.05, 0.95, model['equation'], transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=10)
    
    plt.tight_layout()
    
    # Save calibration plot
    calibration_plot_path = OUTPUT_DIR / 'calibration_models.png'
    fig.savefig(calibration_plot_path, dpi=300, bbox_inches='tight')
    print(f"Calibration plot saved: {calibration_plot_path}")
    
    plt.show()
else:
    print("No calibration models to plot.")

## Apply Calibrations to All Data

In [ ]:
def apply_calibration(satellite_df, calibration_model):
    """
    Apply calibration model to satellite data.
    
    Args:
        satellite_df: DataFrame with satellite observations
        calibration_model: Calibration model dictionary
    
    Returns:
        DataFrame with calibrated chlorophyll values
    """
    if not calibration_model or not calibration_model['success']:
        return satellite_df.copy()
    
    df = satellite_df.copy()
    x = df['value'].values
    
    if calibration_model['model_type'] == 'linear':
        df['calibrated_chl'] = linear_model(x, *calibration_model['params'])
    elif calibration_model['model_type'] == 'exponential':
        df['calibrated_chl'] = exponential_model(x, *calibration_model['params'])
    elif calibration_model['model_type'] == 'power':
        df['calibrated_chl'] = power_model(x, *calibration_model['params'])
    
    # Ensure non-negative values
    df['calibrated_chl'] = np.maximum(df['calibrated_chl'], 0)
    
    return df

# Apply calibrations to all datasets
print("Applying calibrations to satellite data...\n")

calibrated_data = {}

# Upper Klamath Lake
if 'sentinel' in calibration_models:
    calibrated_data['ukl_sentinel'] = apply_calibration(ukl_sentinel, calibration_models['sentinel'])
    print(f"Applied Sentinel-2 calibration to UKL data ({len(calibrated_data['ukl_sentinel'])} points)")

if 'modis_aqua' in calibration_models:
    calibrated_data['ukl_modis_aqua'] = apply_calibration(ukl_modis_aqua, calibration_models['modis_aqua'])
    print(f"Applied MODIS-Aqua calibration to UKL data ({len(calibrated_data['ukl_modis_aqua'])} points)")

if 'modis_terra' in calibration_models:
    calibrated_data['ukl_modis_terra'] = apply_calibration(ukl_modis_terra, calibration_models['modis_terra'])
    print(f"Applied MODIS-Terra calibration to UKL data ({len(calibrated_data['ukl_modis_terra'])} points)")

# Detroit Lake
if 'sentinel' in calibration_models:
    calibrated_data['detroit_sentinel'] = apply_calibration(detroit_sentinel, calibration_models['sentinel'])
    print(f"Applied Sentinel-2 calibration to Detroit data ({len(calibrated_data['detroit_sentinel'])} points)")

if 'modis_aqua' in calibration_models:
    calibrated_data['detroit_modis_aqua'] = apply_calibration(detroit_modis_aqua, calibration_models['modis_aqua'])
    print(f"Applied MODIS-Aqua calibration to Detroit data ({len(calibrated_data['detroit_modis_aqua'])} points)")

if 'modis_terra' in calibration_models:
    calibrated_data['detroit_modis_terra'] = apply_calibration(detroit_modis_terra, calibration_models['modis_terra'])
    print(f"Applied MODIS-Terra calibration to Detroit data ({len(calibrated_data['detroit_modis_terra'])} points)")

print(f"\nCalibration applied to {len(calibrated_data)} datasets.")

## Comprehensive Time Series Visualization

In [ ]:
# Create comprehensive time series plots
def create_time_series_plot(data_dict, title, save_name, include_insitu=None):
    """
    Create time series plot for multiple sensors.
    """
    fig, ax = plt.subplots(figsize=(15, 8))
    
    colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']
    color_idx = 0
    
    for name, df in data_dict.items():
        if len(df) > 0:
            if 'calibrated_chl' in df.columns:
                y_values = df['calibrated_chl']
                label = f"{name.replace('_', ' ').title()} (Calibrated)"
            else:
                y_values = df['value']
                label = f"{name.replace('_', ' ').title()}"
            
            ax.scatter(df['date'], y_values, 
                      color=colors[color_idx % len(colors)], 
                      alpha=0.6, s=20, label=label)
            color_idx += 1
    
    # Add in situ data if provided
    if include_insitu is not None and len(include_insitu) > 0:
        ax.scatter(include_insitu['date'], include_insitu['chlorophyll_ugL'],
                  color='black', s=30, marker='o', 
                  label='In Situ Observations', alpha=0.8)
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Chlorophyll-a (µg/L)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = OUTPUT_DIR / f'{save_name}.png'
    fig.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Time series plot saved: {plot_path}")
    
    return fig

# Upper Klamath Lake time series
ukl_data = {k: v for k, v in calibrated_data.items() if 'ukl' in k}
if ukl_data:
    fig_ukl = create_time_series_plot(
        ukl_data, 
        'Upper Klamath Lake - Calibrated Satellite Chlorophyll Time Series',
        'UKL_calibrated_timeseries',
        include_insitu=ukl_insitu
    )
    plt.show()

# Detroit Lake time series
detroit_data = {k: v for k, v in calibrated_data.items() if 'detroit' in k}
if detroit_data:
    fig_detroit = create_time_series_plot(
        detroit_data,
        'Detroit Lake - Calibrated Satellite Chlorophyll Time Series',
        'Detroit_calibrated_timeseries'
    )
    plt.show()

# Combined comparison plot
if ukl_data and detroit_data:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
    
    # UKL subplot
    colors = ['blue', 'green', 'red']
    for i, (name, df) in enumerate(ukl_data.items()):
        if len(df) > 0 and 'calibrated_chl' in df.columns:
            ax1.scatter(df['date'], df['calibrated_chl'], 
                       color=colors[i % len(colors)], alpha=0.6, s=20,
                       label=name.replace('ukl_', '').replace('_', '-').title())
    
    if len(ukl_insitu) > 0:
        ax1.scatter(ukl_insitu['date'], ukl_insitu['chlorophyll_ugL'],
                   color='black', s=30, alpha=0.8, label='In Situ')
    
    ax1.set_ylabel('Chlorophyll-a (µg/L)')
    ax1.set_title('Upper Klamath Lake')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Detroit subplot
    for i, (name, df) in enumerate(detroit_data.items()):
        if len(df) > 0 and 'calibrated_chl' in df.columns:
            ax2.scatter(df['date'], df['calibrated_chl'], 
                       color=colors[i % len(colors)], alpha=0.6, s=20,
                       label=name.replace('detroit_', '').replace('_', '-').title())
    
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Chlorophyll-a (µg/L)')
    ax2.set_title('Detroit Lake')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    
    comparison_path = OUTPUT_DIR / 'lake_comparison_timeseries.png'
    fig.savefig(comparison_path, dpi=300, bbox_inches='tight')
    print(f"Comparison plot saved: {comparison_path}")
    
    plt.show()

## Statistical Analysis and Goodness of Fit

In [ ]:
# Comprehensive statistical analysis
def calculate_comprehensive_stats(y_true, y_pred, sensor_name="Unknown"):
    """
    Calculate comprehensive statistical metrics.
    """
    n = len(y_true)
    
    # Basic statistics
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    
    # Relative statistics
    mean_true = np.mean(y_true)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rrmse = rmse / mean_true * 100
    
    # Correlation
    correlation, p_value = stats.pearsonr(y_true, y_pred)
    
    # Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - mean_true) ** 2))
    
    # Index of Agreement (Willmott)
    d = 1 - (np.sum((y_true - y_pred) ** 2) / 
             np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2))
    
    # Slope and intercept of best fit line
    slope, intercept, r_value, p_value_reg, std_err = stats.linregress(y_true, y_pred)
    
    return {
        'sensor': sensor_name,
        'n': n,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'bias': bias,
        'mape': mape,
        'rrmse': rrmse,
        'correlation': correlation,
        'p_value': p_value,
        'nse': nse,
        'willmott_d': d,
        'slope': slope,
        'intercept': intercept,
        'mean_observed': mean_true,
        'mean_predicted': np.mean(y_pred),
        'std_observed': np.std(y_true),
        'std_predicted': np.std(y_pred)
    }

# Calculate statistics for calibration datasets
print("Statistical Analysis of Calibration Performance\n")
print("=" * 80)

calibration_stats = []

for sensor_key, match_df in matches.items():
    if len(match_df) > 0 and sensor_key in calibration_models:
        y_true = match_df['insitu_chl'].values
        
        # Apply calibration model to get predictions
        x = match_df['satellite_value'].values
        model = calibration_models[sensor_key]
        
        if model['model_type'] == 'linear':
            y_pred = linear_model(x, *model['params'])
        elif model['model_type'] == 'exponential':
            y_pred = exponential_model(x, *model['params'])
        elif model['model_type'] == 'power':
            y_pred = power_model(x, *model['params'])
        
        stats_dict = calculate_comprehensive_stats(y_true, y_pred, sensor_key)
        calibration_stats.append(stats_dict)
        
        print(f"{sensor_key.upper()}:")
        print(f"  Model Type: {model['model_type']}")
        print(f"  Number of points: {stats_dict['n']}")
        print(f"  R²: {stats_dict['r2']:.3f}")
        print(f"  RMSE: {stats_dict['rmse']:.2f} µg/L")
        print(f"  MAE: {stats_dict['mae']:.2f} µg/L")
        print(f"  Bias: {stats_dict['bias']:.2f} µg/L")
        print(f"  MAPE: {stats_dict['mape']:.1f}%")
        print(f"  Relative RMSE: {stats_dict['rrmse']:.1f}%")
        print(f"  Nash-Sutcliffe Efficiency: {stats_dict['nse']:.3f}")
        print(f"  Willmott Index: {stats_dict['willmott_d']:.3f}")
        print(f"  Correlation: {stats_dict['correlation']:.3f} (p={stats_dict['p_value']:.3f})")
        print()

# Create summary statistics table
if calibration_stats:
    stats_df = pd.DataFrame(calibration_stats)
    
    # Save to CSV
    stats_csv_path = OUTPUT_DIR / 'calibration_statistics.csv'
    stats_df.to_csv(stats_csv_path, index=False)
    print(f"Calibration statistics saved: {stats_csv_path}")
    
    # Display summary table
    print("\nSUMMARY STATISTICS TABLE")
    print("=" * 80)
    
    summary_cols = ['sensor', 'n', 'r2', 'rmse', 'mae', 'mape', 'nse']
    print(stats_df[summary_cols].round(3).to_string(index=False))

## Cross-Validation Analysis

In [ ]:
# Cross-validation analysis
def perform_cross_validation(x, y, sensor_name, cv_type='kfold', k=5):
    """
    Perform cross-validation analysis.
    
    Args:
        x: Satellite values
        y: In situ values
        sensor_name: Name for reporting
        cv_type: 'kfold' or 'loo' (leave-one-out)
        k: Number of folds for k-fold CV
    
    Returns:
        Dictionary with CV results
    """
    if len(x) < 3:
        return None
    
    # Prepare data
    X = x.reshape(-1, 1)
    
    # Choose cross-validation method
    if cv_type == 'loo':
        cv = LeaveOneOut()
        cv_name = "Leave-One-Out"
    else:
        cv = KFold(n_splits=min(k, len(x)), shuffle=True, random_state=42)
        cv_name = f"{k}-Fold"
    
    # Models to test
    models = {
        'Linear': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
    }
    
    results = {
        'sensor': sensor_name,
        'cv_type': cv_name,
        'n_samples': len(x),
        'models': {}
    }
    
    for model_name, model in models.items():
        try:
            # Perform cross-validation
            cv_scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
            cv_rmse = np.sqrt(-cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error'))
            
            results['models'][model_name] = {
                'r2_scores': cv_scores,
                'r2_mean': cv_scores.mean(),
                'r2_std': cv_scores.std(),
                'rmse_scores': cv_rmse,
                'rmse_mean': cv_rmse.mean(),
                'rmse_std': cv_rmse.std()
            }
            
        except Exception as e:
            print(f"CV failed for {sensor_name} - {model_name}: {e}")
    
    return results

print("Cross-Validation Analysis\n")
print("=" * 60)

cv_results = []

for sensor_key, match_df in matches.items():
    if len(match_df) >= 5:  # Need at least 5 points for meaningful CV
        x = match_df['satellite_value'].values
        y = match_df['insitu_chl'].values
        
        print(f"\n{sensor_key.upper()} Cross-Validation:")
        
        # K-fold cross-validation
        kfold_results = perform_cross_validation(x, y, sensor_key, 'kfold', 5)
        if kfold_results:
            cv_results.append(kfold_results)
            
            print(f"  5-Fold Cross-Validation Results:")
            for model_name, model_results in kfold_results['models'].items():
                print(f"    {model_name}:")
                print(f"      R² = {model_results['r2_mean']:.3f} ± {model_results['r2_std']:.3f}")
                print(f"      RMSE = {model_results['rmse_mean']:.2f} ± {model_results['rmse_std']:.2f} µg/L")
        
        # Leave-one-out cross-validation (if not too many points)
        if len(match_df) <= 50:
            loo_results = perform_cross_validation(x, y, sensor_key, 'loo')
            if loo_results:
                print(f"  Leave-One-Out Cross-Validation Results:")
                for model_name, model_results in loo_results['models'].items():
                    print(f"    {model_name}:")
                    print(f"      R² = {model_results['r2_mean']:.3f} ± {model_results['r2_std']:.3f}")
                    print(f"      RMSE = {model_results['rmse_mean']:.2f} ± {model_results['rmse_std']:.2f} µg/L")
        else:
            print(f"  Skipping Leave-One-Out CV (too many points: {len(match_df)})")
    
    else:
        print(f"\n{sensor_key.upper()}: Insufficient data for cross-validation (need ≥5, have {len(match_df)})")

# Save cross-validation results
if cv_results:
    # Flatten results for CSV export
    cv_flat = []
    for result in cv_results:
        for model_name, model_data in result['models'].items():
            cv_flat.append({
                'sensor': result['sensor'],
                'cv_type': result['cv_type'],
                'n_samples': result['n_samples'],
                'model': model_name,
                'r2_mean': model_data['r2_mean'],
                'r2_std': model_data['r2_std'],
                'rmse_mean': model_data['rmse_mean'],
                'rmse_std': model_data['rmse_std']
            })
    
    cv_df = pd.DataFrame(cv_flat)
    cv_csv_path = OUTPUT_DIR / 'cross_validation_results.csv'
    cv_df.to_csv(cv_csv_path, index=False)
    print(f"\nCross-validation results saved: {cv_csv_path}")

print("\nCross-validation analysis complete.")

## Export Calibrated Data

In [ ]:
# Export calibrated datasets
print("Exporting calibrated datasets...\n")

export_dir = OUTPUT_DIR / 'calibrated_data'
export_dir.mkdir(exist_ok=True)

for dataset_name, df in calibrated_data.items():
    if len(df) > 0 and 'calibrated_chl' in df.columns:
        # Prepare export dataframe
        export_df = df[['date', 'value', 'calibrated_chl', 'sensor']].copy()
        export_df.columns = ['date', 'satellite_value', 'calibrated_chlorophyll_ugL', 'sensor']
        
        # Add metadata
        if dataset_name.split('_')[1] in calibration_models:
            sensor_key = dataset_name.split('_')[1]
            if len(dataset_name.split('_')) > 2:
                sensor_key = '_'.join(dataset_name.split('_')[1:])
            
            model = calibration_models.get(sensor_key, {})
            export_df['calibration_model'] = model.get('model_type', 'unknown')
            export_df['calibration_r2'] = model.get('r2', np.nan)
            export_df['calibration_rmse'] = model.get('rmse', np.nan)
        
        # Sort by date
        export_df = export_df.sort_values('date')
        
        # Export to CSV
        export_path = export_dir / f'{dataset_name}_calibrated.csv'
        export_df.to_csv(export_path, index=False)
        print(f"Exported {dataset_name}: {len(export_df)} records → {export_path}")

# Export calibration model parameters
if calibration_models:
    model_params = []
    for sensor, model in calibration_models.items():
        params_dict = {
            'sensor': sensor,
            'model_type': model['model_type'],
            'equation': model['equation'],
            'r2': model['r2'],
            'rmse': model['rmse'],
            'mae': model['mae'],
            'n_points': model['n_points']
        }
        
        # Add parameter values
        for i, param in enumerate(model['params']):
            params_dict[f'param_{i+1}'] = param
        
        model_params.append(params_dict)
    
    model_df = pd.DataFrame(model_params)
    model_path = OUTPUT_DIR / 'calibration_model_parameters.csv'
    model_df.to_csv(model_path, index=False)
    print(f"\nCalibration model parameters saved: {model_path}")

print(f"\nAll calibrated data exported to: {export_dir}")
print(f"Analysis results saved to: {OUTPUT_DIR}")

## Summary Report

In [ ]:
# Generate comprehensive summary report
print("\n" + "=" * 80)
print("SATELLITE CHLOROPHYLL CALIBRATION AND VALIDATION SUMMARY")
print("=" * 80)

print(f"\nAnalysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Matching Tolerance: ±{MATCH_TOLERANCE_DAYS} days")

print("\n1. DATA AVAILABILITY:")
print(f"   Upper Klamath Lake (Calibration Site):")
print(f"     - In situ observations: {len(ukl_insitu)}")
print(f"     - Sentinel-2 observations: {len(ukl_sentinel)}")
print(f"     - MODIS-Aqua observations: {len(ukl_modis_aqua)}")
print(f"     - MODIS-Terra observations: {len(ukl_modis_terra)}")
print(f"   Detroit Lake (Application Site):")
print(f"     - Sentinel-2 observations: {len(detroit_sentinel)}")
print(f"     - MODIS-Aqua observations: {len(detroit_modis_aqua)}")
print(f"     - MODIS-Terra observations: {len(detroit_modis_terra)}")

print("\n2. TEMPORAL MATCHING RESULTS:")
total_matches = sum(len(df) for df in matches.values())
print(f"   Total matched observations: {total_matches}")
for sensor, match_df in matches.items():
    if len(match_df) > 0:
        avg_diff = match_df['days_diff'].mean()
        print(f"   {sensor}: {len(match_df)} matches (avg {avg_diff:.1f} days difference)")

print("\n3. CALIBRATION MODELS DEVELOPED:")
if calibration_models:
    for sensor, model in calibration_models.items():
        print(f"   {sensor}:")
        print(f"     Model: {model['model_type']}")
        print(f"     Equation: {model['equation']}")
        print(f"     Performance: R² = {model['r2']:.3f}, RMSE = {model['rmse']:.2f} µg/L")
        print(f"     Training points: {model['n_points']}")
else:
    print("   No calibration models could be developed")
    print("   Reason: Insufficient matched observations")

print("\n4. CALIBRATED DATASETS:")
for dataset_name, df in calibrated_data.items():
    if 'calibrated_chl' in df.columns:
        lake = 'Upper Klamath Lake' if 'ukl' in dataset_name else 'Detroit Lake'
        sensor = dataset_name.split('_')[-1].replace('_', '-').title()
        if 'modis' in dataset_name:
            sensor = f"MODIS-{sensor}"
        print(f"   {lake} - {sensor}: {len(df)} calibrated observations")

print("\n5. KEY FINDINGS:")
if calibration_stats:
    best_sensor = max(calibration_stats, key=lambda x: x['r2'])
    print(f"   Best performing sensor: {best_sensor['sensor']} (R² = {best_sensor['r2']:.3f})")
    
    avg_r2 = np.mean([s['r2'] for s in calibration_stats])
    print(f"   Average calibration R²: {avg_r2:.3f}")
    
    rmse_range = [s['rmse'] for s in calibration_stats]
    print(f"   RMSE range: {min(rmse_range):.2f} - {max(rmse_range):.2f} µg/L")
else:
    print("   No statistical analysis available (insufficient data)")

print("\n6. RECOMMENDATIONS:")
print("   - Use calibrated chlorophyll values for water quality assessments")
print("   - Consider sensor-specific uncertainties in analysis")
print("   - Update calibrations periodically with new in situ data")
if len(ukl_insitu) == 0 or 'synthetic' in str(ukl_insitu['source'].iloc[0]) if len(ukl_insitu) > 0 else True:
    print("   - IMPORTANT: Replace synthetic data with actual in situ observations")
    print("   - Collect more in situ data for improved calibration")

print("\n7. OUTPUT FILES:")
print(f"   Results directory: {OUTPUT_DIR}")
print(f"   Calibrated data: {OUTPUT_DIR}/calibrated_data/")
print(f"   Calibration plots: {OUTPUT_DIR}/calibration_models.png")
print(f"   Time series plots: {OUTPUT_DIR}/*_timeseries.png")
print(f"   Statistics: {OUTPUT_DIR}/calibration_statistics.csv")
print(f"   Cross-validation: {OUTPUT_DIR}/cross_validation_results.csv")
print(f"   Model parameters: {OUTPUT_DIR}/calibration_model_parameters.csv")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

# Save summary report
summary_path = OUTPUT_DIR / 'analysis_summary.txt'
with open(summary_path, 'w') as f:
    f.write("SATELLITE CHLOROPHYLL CALIBRATION AND VALIDATION SUMMARY\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Matching Tolerance: ±{MATCH_TOLERANCE_DAYS} days\n\n")
    
    if calibration_models:
        f.write("CALIBRATION MODELS:\n")
        for sensor, model in calibration_models.items():
            f.write(f"{sensor}: {model['equation']} (R² = {model['r2']:.3f})\n")
    
    f.write(f"\nTotal calibrated observations: {sum(len(df) for df in calibrated_data.values())}\n")

print(f"\nSummary report saved: {summary_path}")